# DC4 parallel-chunks source catalog: Galactic Aitoff map

Plot every entry in the **full-observation parallel-chunks YAML catalog**. Labeled point sources appear as text without star markers; their labels are shifted as needed to avoid overlap. The four unlabeled point components remain stars. Names are shortened on the map; the 11 GRBs and six MGFs are left unlabeled and use different colored circles. Diffuse line/continuum templates and split CO nova components are also shown without text labels; their full names remain in the table below. The full catalog names appear in the table below. Analytic extended sources show their centers and one characteristic width. For HEALPix templates, the map shows the brightest pixel and the smallest set of pixels containing 68% of the energy-integrated, nonnegative template intensity. The footprint is a visual locator, not a confidence region or detection contour.

**Why 71 entries when DC4 reports 64 sources?** The [DC4 data-products guide](https://github.com/cositools/cosi-data-challenges/blob/main/data-products/README.md) counts distinct astrophysical sources in the mock dataset. This YAML keeps seven extra model entries because some sources were simulated in multiple states or components: 4C+21.35 (2 entries, +1), 3C 454.3 (2, +1), the CO nova (3, +2), and line/continuum pairs for positrons from 26Al, positrons from 44Ti, and the positron thin disk (2 each, +3). Thus 71 catalog entries represent 64 grouped sources. The legend counts physical sources: 29 labeled point sources + 2 other point sources (four components) + 11 GRBs + 6 MGFs + 8 analytic extended sources + 8 HEALPix sources (11 templates) = 64. All 71 entries remain in the YAML and CSV; co-located source states share one name on the map.

The 26Al disk template contains the 1809 keV nuclear line; the 60Fe disk template contains the 1173 and 1333 keV lines. Their energy dependence is stored in the spatial-energy FITS templates, while the catalog `Constant` spectrum is a normalization multiplier.

The figure and exported PNG have transparent backgrounds. White labels and grid lines are intended for a dark slide background.

The 11 HEALPix template FITS files referenced by the YAML must be available. Set `DC4_TEMPLATE_DIR` if they have been moved.

In [53]:
from pathlib import Path
import csv
import os

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import yaml
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits

# Use the requested catalog. This works from either this notebook's folder
# or the cosipy repository root; an environment override supports other layouts.
CATALOG_NAME = 'source_catalog_DC4_all_71files_full_elapsed_lightcurve_parallel_chunks_generated.yaml'
CATALOG_PATH = Path(os.environ.get('DC4_CATALOG_YAML', CATALOG_NAME)).expanduser()
if not CATALOG_PATH.exists():
    CATALOG_PATH = Path.cwd() / 'docs/tutorials/spectral_fits/continuum_fit/AGN' / CATALOG_NAME
if not CATALOG_PATH.is_file():
    raise FileNotFoundError(f'Cannot find catalog: {CATALOG_PATH}')

TEMPLATE_DIR = os.environ.get('DC4_TEMPLATE_DIR')  # Optional replacement for YAML FITS directory.
SAVE_FIGURE = True  # Write a transparent PNG for slides.
FIGURE_PATH = CATALOG_PATH.with_name('DC4_Source_Catalog_Aitoff_Map.png')
CSV_PATH = CATALOG_PATH.with_name('DC4_Source_Catalog_Aitoff_Sources.csv')
LABEL_FONT_SIZE = 20  # Increase this for larger source-name labels (rerun the plot cell).

with CATALOG_PATH.open() as stream:
    catalog = yaml.safe_load(stream)
assert isinstance(catalog, dict) and len(catalog) == 71, 'Expected the 71-source parallel-chunks catalog.'
print(f'Loaded {len(catalog)} sources from {CATALOG_PATH}')

Loaded 71 sources from source_catalog_DC4_all_71files_full_elapsed_lightcurve_parallel_chunks_generated.yaml


## Read sky positions and template footprints

Energy integration here locates the spatial templates for display; it does not perform detector-response folding or fit a spectrum.

In [54]:
def template_location(model):
    # The catalog records an absolute FITS path. A relocated template folder
    # may be supplied without editing the scientific catalog.
    original = Path(model['extra_setup']['_fitsfile']).expanduser()
    path = Path(TEMPLATE_DIR).expanduser() / original.name if TEMPLATE_DIR else original
    if not path.is_file():
        raise FileNotFoundError(f'Missing spatial template: {path}')
    with fits.open(path, memmap=True) as hdus:
        skymap = hdus['SKYMAP'].data
        energies = np.asarray(hdus['ENERGIES'].data['ENERGY'], dtype=float)
        layers = np.column_stack([np.asarray(skymap.field(i), dtype=float) for i in range(len(energies))])
        ordering = hdus['SKYMAP'].header.get('ORDERING', 'RING').strip().upper()
    if len(energies) > 1:
        # Integrate at the template's native energy knots. Negative or
        # nonfinite intensities cannot contribute to the display footprint.
        weights = np.diff(energies)
        brightness = np.sum((layers[:, :-1] + layers[:, 1:]) * weights[None, :] / 2, axis=1)
    else:
        brightness = layers[:, 0]
    brightness = np.clip(np.nan_to_num(brightness), 0, None)
    if brightness.sum() <= 0:
        raise ValueError(f'Spatial template has no positive intensity: {path}')
    nside = hp.npix2nside(len(brightness))
    lon, lat = hp.pix2ang(nside, np.arange(len(brightness)), nest=ordering.startswith('NEST'), lonlat=True)
    peak = int(np.argmax(brightness))
    ranked = np.argsort(brightness)[::-1]
    count = np.searchsorted(np.cumsum(brightness[ranked]), 0.68 * brightness.sum()) + 1
    footprint = ranked[:count]
    return float(lon[peak]), float(lat[peak]), lon[footprint], lat[footprint]

sources = []
for key, source in catalog.items():
    name = key.rsplit(' (', 1)[0]
    if 'position' in source:
        lon = float(source['position']['l']['value'])
        lat = float(source['position']['b']['value'])
        sources.append(dict(name=name, type='Point source', lon=lon, lat=lat))
    elif 'Gaussian_on_sphere' in source or 'Asymm_Gaussian_on_sphere' in source:
        shape = source.get('Gaussian_on_sphere', source.get('Asymm_Gaussian_on_sphere'))
        lon, lat = float(shape['lon0']['value']), float(shape['lat0']['value'])
        width = float(shape.get('sigma', shape.get('a'))['value'])
        sources.append(dict(name=name, type='Analytic extended', lon=lon, lat=lat, width=width))
    elif 'GalpropHealpixModel' in source:
        lon, lat, footprint_lon, footprint_lat = template_location(source['GalpropHealpixModel'])
        sources.append(dict(name=name, type='HEALPix template', lon=lon, lat=lat,
                            footprint_lon=footprint_lon, footprint_lat=footprint_lat))
    else:
        raise ValueError(f'Unknown spatial model for {key}')

from collections import Counter
print(Counter(item['type'] for item in sources))
assert len(sources) == len(catalog)

Counter({'Point source': 52, 'HEALPix template': 11, 'Analytic extended': 8})


In [ ]:
from matplotlib.font_manager import FontProperties
from matplotlib.transforms import Bbox


def aitoff_xy(lon, lat):
    # Matplotlib's Aitoff x axis grows to the right. Flip it so Galactic
    # longitude increases to the left, as in common sky maps.
    wrapped = (np.asarray(lon) + 180) % 360 - 180
    return -np.deg2rad(wrapped), np.deg2rad(lat)


def draw_width(ax, item, color):
    # A spherical circle at one Gaussian width. Break segments at the
    # longitude seam so an outline cannot span across the whole map.
    center = SkyCoord(l=item['lon'] * u.deg, b=item['lat'] * u.deg, frame='galactic')
    ring = center.directional_offset_by(np.linspace(0, 360, 181) * u.deg, item['width'] * u.deg)
    x, y = aitoff_xy(ring.l.deg, ring.b.deg)
    breaks = np.flatnonzero(np.abs(np.diff(x)) > np.pi) + 1
    for xx, yy in zip(np.split(x, breaks), np.split(y, breaks)):
        ax.plot(xx, yy, color=color, alpha=0.65, linewidth=1)


def is_grb(item):
    return item['name'].lower().startswith('grb_')


def is_mgf(item):
    return item['name'].lower().startswith('mgf_')


# Keep short plot labels unique and recognizable. Full YAML identifiers remain
# in the table below, so abbreviations do not hide any catalog entries.
SHORT_LABELS = {
    'cyg_x1_hard': 'Cyg X-1',
    'one_e_1740_compow': '1E 1740',
    'grs_1758_258': 'GRS 1758',
    'cena': 'Cen A',
    'four_c_71p07': '4C 71.07',
    'four_c_21p35_noflare': '4C+21.35',
    'three_c_454p3_low': '3C 454.3',
    'ngc1068': 'NGC 1068',
    'ngc4151': 'NGC 4151',
    'one_rxs_j170849': '1RXS J170849',
    'generic_magnetar': 'Magnetar',
    'casa_g16_distribution': 'Cas A',
    'mgt_burst_bright_complex': 'Magnetar burst',
    'co_nova_continuum': 'CO nova cont',
    'co_nova_511kev': 'CO nova 511',
    'co_nova_478kev': 'CO nova 478',
    'positrons_26al_line': '26Al e+ line',
    'positrons_26al_cont': '26Al e+ cont',
    'positrons_44ti_line': '44Ti e+ line',
    'positrons_44ti_cont': '44Ti e+ cont',
    'narrow_bulge_511': '511 narrow bulge',
    'broad_bulge_511': '511 broad bulge',
    'vela_snr_511': 'Vela 511',
    'al26_cyg_region': '26Al Cygnus',
    'fe60_cyg_region': '60Fe Cygnus',
    'al26_ne2001': '26Al disk',
    'fe60_ne2001': '60Fe disk',
    'positron_annihilation_in_flight': 'e+ in flight',
    'positrons_central_source': '511 center',
    'positrons_thin_disk_line': '511 thin disk',
    'positrons_thin_disk_cont': 'e+ thin disk cont',
    'orion_eridanus': 'Orion-Eridanus',
    'upper_scorpius': 'Upper Sco',
    'vela_jr_44ti': 'Vela Jr 44Ti',
    'lmc_511': 'LMC 511',
    'm31_511': 'M31 511',
    'virgo_511': 'Virgo 511',
}


def short_label(item):
    return SHORT_LABELS.get(item['name'], item['name'].replace('_', ' '))


# Keep all diffuse line/continuum model markers but omit their names.
# 26Al and 60Fe disk/Cygnus templates are nuclear-line emission, not
# continuum. The extended-source entries also include 511 keV line and
# annihilation-continuum templates. Full identifiers remain in the table.
# CO nova's separate spectral components and the 511 keV central component
# are point entries, so name them explicitly here.
UNLABELED_POINT_COMPONENTS = {
    'co_nova_continuum', 'co_nova_511kev', 'co_nova_478kev',
    'positrons_central_source',
}
# Each state remains in the catalog and CSV. Its shared sky position
# needs just one source name on the map.
SECOND_STATES = {'four_c_21p35_flare', 'three_c_454p3_high'}


def should_label(item):
    return (item['type'] == 'Point source'
            and not (is_grb(item) or is_mgf(item))
            and item['name'] not in UNLABELED_POINT_COMPONENTS
            and item['name'] not in SECOND_STATES)


fig = plt.figure(figsize=(22, 13), facecolor='none')
ax = fig.add_subplot(111, projection='aitoff')
# Set the final axes geometry before measuring labels in display pixels.
fig.subplots_adjust(left=0.04, right=0.96, top=0.90, bottom=0.16)
ax.set_facecolor('none')  # Keep the Aitoff area transparent too.
ax.grid(True, color='white', alpha=0.22, linewidth=0.7)
ax.tick_params(colors='white', labelsize=12)
# Tick positions are projection x coordinates; labels show Galactic longitude.
ax.set_xticks(np.deg2rad(np.arange(-150, 151, 30)))
ax.set_xticklabels(['150°', '120°', '90°', '60°', '30°', '0°',
                    '330°', '300°', '270°', '240°', '210°'])
for spine in ax.spines.values():
    spine.set_edgecolor('#cbd5e1')

colors = {'Point source': '#4dd7f4', 'Analytic extended': '#ffb454',
          'HEALPix template': '#f394e8'}
for item in sources:
    color = colors[item['type']]
    if item['type'] == 'HEALPix template':
        xx, yy = aitoff_xy(item['footprint_lon'], item['footprint_lat'])
        ax.scatter(xx, yy, s=15, marker='s', color=color, alpha=0.07,
                   linewidths=0, rasterized=True)
    elif item['type'] == 'Analytic extended':
        draw_width(ax, item, color)

# Other point sources are stars. GRBs and MGFs use equal-size circular
# markers but distinct colors, and neither transient group has name labels.
def marker_group(item):
    if item['type'] != 'Point source':
        return item['type']
    if is_grb(item):
        return 'GRB'
    if is_mgf(item):
        return 'MGF'
    if item['name'] in SECOND_STATES:
        return 'Shared source position'
    # Labeled sources are drawn only with text; retain stars for the four
    # unlabeled CO nova / positron point components.
    return 'Labeled point source' if should_label(item) else 'Unlabeled point component'


point_styles = [
    ('Unlabeled point component', '*', 100, colors['Point source'],
     'Other point sources (2): 4 components'),
    ('GRB', 'o', 22, '#73c7ff', 'GRBs (11)'),
    ('MGF', 'o', 22, '#ff7272', 'MGFs (6)'),
    ('Analytic extended', 'o', 55, colors['Analytic extended'],
     'Analytic extended sources (8)'),
    ('HEALPix template', 'D', 46, colors['HEALPix template'],
     'HEALPix sources (8): 11 templates'),
]
for group, marker, size, color, label in point_styles:
    subset = [item for item in sources if marker_group(item) == group]
    x, y = aitoff_xy([item['lon'] for item in subset], [item['lat'] for item in subset])
    hollow = group == 'Analytic extended'
    ax.scatter(x, y, s=size, marker=marker,
               facecolors='none' if hollow else color,
               edgecolors=color if hollow else '#182132',
               linewidths=1.3 if hollow else 0.5, zorder=4, label=label)

# Label discrete point sources, shifting crowded names to avoid collisions.
# Source coordinates are unchanged; leader lines connect distant labels.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
font = FontProperties(size=LABEL_FONT_SIZE)
occupied = []
labelled = [item for item in sources if should_label(item)]
assert len(labelled) == 29
assert len({short_label(item) for item in labelled}) == len(labelled), 'Plot labels must be unique.'
# Place the most crowded locations first, where placement is least flexible.
xy = {id(item): ax.transData.transform(aitoff_xy(item['lon'], item['lat']))
      for item in labelled}
all_labelled = tuple(labelled)
labelled.sort(key=lambda item: sum(np.linalg.norm(xy[id(item)] - xy[id(other)]) < 95
                                   for other in all_labelled if other is not item), reverse=True)
point_size = fig.dpi / 72
candidates = [(radius * sx, radius * sy)
              for radius in (7, 16, 28, 42, 60, 82, 110, 145)
              for sx, sy in ((1, 1), (-1, 1), (1, -1), (-1, -1),
                             (1, 0), (-1, 0), (0, 1), (0, -1))]
for item in labelled:
    name = short_label(item)
    anchor_x, anchor_y = xy[id(item)]
    width, height, _ = renderer.get_text_width_height_descent(name, font, ismath=False)
    best = None
    for dx, dy in candidates:
        x = anchor_x + dx * point_size
        y = anchor_y + dy * point_size
        left = x if dx >= 0 else x - width
        bottom = y if dy >= 0 else y - height
        box = Bbox.from_bounds(left - 2, bottom - 2, width + 4, height + 4)
        # Keep labels within the dark figure, clear of the legend.
        outside = (box.x0 < 7 or box.x1 > fig.bbox.x1 - 7 or
                   box.y0 < fig.bbox.y0 + 88 or box.y1 > fig.bbox.y1 - 75)
        if outside:
            continue
        overlap = sum(box.overlaps(previous) for previous in occupied)
        score = (overlap, abs(dx) + abs(dy))
        if best is None or score < best[0]:
            best = (score, dx, dy, box)
            if overlap == 0:
                break
    if best is None:
        raise RuntimeError(f'Cannot place label for {name}')
    _, dx, dy, box = best
    occupied.append(box)
    ax.annotate(name, (float(aitoff_xy(item['lon'], item['lat'])[0]),
                       float(aitoff_xy(item['lon'], item['lat'])[1])),
                xytext=(dx, dy), textcoords='offset points',
                ha='left' if dx >= 0 else 'right',
                va='bottom' if dy >= 0 else 'top',
                fontsize=LABEL_FONT_SIZE, color='white', zorder=5,
                arrowprops=dict(arrowstyle='-', color=colors[item['type']],
                                lw=0.45, alpha=0.55, shrinkA=2, shrinkB=3)
                if abs(dx) + abs(dy) > 25 else None)

# ax.set_title('DC4 parallel-chunks source catalog — Galactic Aitoff projection',
            #  color='white', fontsize=20, pad=28)
ax.set_xlabel('Galactic longitude', color='white', fontsize=17, labelpad=22)
ax.set_ylabel('Galactic latitude', color='white', fontsize=17, labelpad=26)
# Legend numbers count physical sources, so they sum to 64. The four
# unlabeled point entries are the three CO nova components plus one
# central positron source; three HEALPix line/continuum pairs share names.
from matplotlib.lines import Line2D
handles, legend_labels = ax.get_legend_handles_labels()
handles.insert(0, Line2D([], [], linestyle='None', color='none'))
legend_labels.insert(0, 'Labeled point sources (29): text')
assert 29 + 2 + 11 + 6 + 8 + 8 == 64
ax.legend(handles, legend_labels, loc='lower center', bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=10,
          facecolor='#101827', edgecolor='#556378', framealpha=1, labelcolor='white')
if SAVE_FIGURE:
    fig.savefig(FIGURE_PATH, dpi=220, transparent=True, bbox_inches='tight')
    print(f'Saved {FIGURE_PATH}')
plt.show()


## All catalog entries

The following cell prints and saves the same four-column list as a CSV. For HEALPix templates, the coordinates are the brightest pixel of the energy-integrated template; they are not fitted source positions.

In [ ]:
# Print and export the same alphabetically sorted source list. For map
# templates, l/b identify the brightest energy-integrated HEALPix pixel.
source_rows = sorted(sources, key=lambda row: row['name'].lower())
print(f"{'Source':40} {'Type':19} {'l (deg)':>9} {'b (deg)':>9}")
print('-' * 82)
for item in source_rows:
    print(f"{item['name']:40} {item['type']:19} {item['lon']:9.2f} {item['lat']:9.2f}")
with CSV_PATH.open('w', newline='') as stream:
    writer = csv.writer(stream, lineterminator="\n")
    writer.writerow(['Source', 'Type', 'l (deg)', 'b (deg)'])
    for item in source_rows:
        writer.writerow([item['name'], item['type'], f"{item['lon']:.2f}", f"{item['lat']:.2f}"])
print(f'Saved {len(source_rows)} entries to {CSV_PATH}')
# The DC4 guide counts physical sources; this catalog retains separate states
# and emission components. Verify the seven extra entries explicitly.
shared_source_groups = {
    '4C+21.35': ['four_c_21p35_noflare', 'four_c_21p35_flare'],
    '3C 454.3': ['three_c_454p3_low', 'three_c_454p3_high'],
    'CO nova': ['co_nova_continuum', 'co_nova_511kev', 'co_nova_478kev'],
    '26Al positrons': ['positrons_26al_line', 'positrons_26al_cont'],
    '44Ti positrons': ['positrons_44ti_line', 'positrons_44ti_cont'],
    'Thin-disk positrons': ['positrons_thin_disk_line', 'positrons_thin_disk_cont'],
}
all_names = {item['name'] for item in sources}
assert all(set(members) <= all_names for members in shared_source_groups.values())
extra_entries = sum(len(members) - 1 for members in shared_source_groups.values())
print(f"\n{len(sources)} YAML entries - {extra_entries} extra state/component entries "
      f"= {len(sources) - extra_entries} grouped astrophysical sources")
assert len(sources) - extra_entries == 64


Source                                   Type                  l (deg)   b (deg)
----------------------------------------------------------------------------------
al26_cyg_region                          Analytic extended       81.00      1.00
al26_ne2001                              HEALPix template        16.88      0.00
broad_bulge_511                          HEALPix template         5.62      0.00
casa_g16_distribution                    Point source           111.73     -2.13
cena                                     Point source           309.52     19.42
co_nova_478kev                           Point source           310.98      2.73
co_nova_511kev                           Point source           310.98      2.73
co_nova_continuum                        Point source           310.98      2.73
crab                                     Point source           184.56     -5.78
cyg_x1_hard                              Point source            71.33      3.07
cyg_x3                    